TASK1

In [11]:
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

end_date = pd.Timestamp.now()
start_date = end_date - pd.DateOffset(years=3)

np.random.seed(42)
n = 500_000
df = pd.DataFrame({
    "user_id": np.arange(1,n+1),
    "city": np.random.choice(["Berlin", "Seoul", "Nairobi", "Toronto", "Lima","Tokyo", "New York","Baku","Ankara","London"], n),
    "score": np.random.uniform(0, 100, n),
    "active": np.random.choice([True, False], n),
    "signup_date": pd.to_datetime(np.random.uniform(start_date.value, end_date.value,n)),
    "age": np.random.randint(18, 81, n),
    "sessions": np.random.randint(0, 501, n),
    "revenue": np.random.uniform(0, 1000, n),
})

In [12]:
df

,user_id,city,score,active,signup_date,age,sessions,revenue
0,1,New York,43.689490,False,2025-12-11 00:45:44.905779200,42,359,272.618107
1,2,Toronto,97.403462,True,2025-03-03 03:42:39.456900608,28,264,107.594249
2,3,Baku,66.148238,False,2024-10-18 07:37:11.266229760,27,480,372.380026
3,4,Lima,21.993544,False,2025-06-28 14:47:27.135960832,75,330,879.705424
4,5,New York,91.719953,True,2024-01-03 14:53:11.526349568,79,140,730.620245
...,...,...,...,...,...,...,...,...
499995,499996,Toronto,61.531318,False,2025-06-05 20:16:35.967020800,77,5,915.694460
499996,499997,Berlin,74.712924,False,2024-02-26 10:38:03.815138816,38,418,904.013702
499997,499998,Ankara,48.430697,True,2023-11-07 13:36:41.184132352,28,233,423.253865
499998,499999,Baku,89.680757,True,2024-10-29 11:15:45.442146560,23,383,626.845396


In [13]:
#step2

# Write to Parquet
df.to_parquet("data.parquet", index=False)

In [14]:
# Inspect metadata
parquet_file = pq.ParquetFile("data.parquet")
print(f"Number of row groups: {parquet_file.metadata.num_row_groups}")
print(f"Number of columns: {parquet_file.metadata.num_columns}")
print(f"Number of rows: {parquet_file.metadata.num_rows}")
print(f"\nSchema:")
print(parquet_file.schema_arrow)

Number of row groups: 1
Number of columns: 8
Number of rows: 500000

Schema:
user_id: int64
city: string
score: double
active: bool
signup_date: timestamp[ns]
age: int32
sessions: int32
revenue: double
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 992


In [15]:
row_group = parquet_file.metadata.row_group(0)

for i in range(row_group.num_columns):
    column = row_group.column(i)
    stats = column.statistics

    print(f"Column: {column.path_in_schema}")
    print(f"  Physical Type: {column.physical_type}")
    print(f"  Compressed Size: {column.total_compressed_size} bytes")

    if stats:
        print(f"  Min: {stats.min}")
        print(f"  Max: {stats.max}")
        print(f"  Null Count: {stats.null_count}")
    else:
        print("  No statistics available")

    print("-" * 40)

Column: user_id
  Physical Type: INT64
  Compressed Size: 2273849 bytes
  Min: 1
  Max: 500000
  Null Count: 0
----------------------------------------
Column: city
  Physical Type: BYTE_ARRAY
  Compressed Size: 252611 bytes
  Min: Ankara
  Max: Toronto
  Null Count: 0
----------------------------------------
Column: score
  Physical Type: DOUBLE
  Compressed Size: 4272959 bytes
  Min: 5.188445665327279e-05
  Max: 99.99983148609545
  Null Count: 0
----------------------------------------
Column: active
  Physical Type: BOOLEAN
  Compressed Size: 63800 bytes
  Min: False
  Max: True
  Null Count: 0
----------------------------------------
Column: signup_date
  Physical Type: INT64
  Compressed Size: 4272963 bytes
  Min: 2023-03-08 18:59:47.740241920
  Max: 2026-03-08 18:58:13.498848256
  Null Count: 0
----------------------------------------
Column: age
  Physical Type: INT32
  Compressed Size: 377947 bytes
  Min: 18
  Max: 80
  Null Count: 0
----------------------------------------
Col

In [16]:
#step3
df.to_csv("data.csv", index=False)

In [17]:
from pathlib import Path
parquet_path = Path("data.parquet")
csv_path = Path("data.csv")
parquet_size = parquet_path.stat().st_size
csv_size = csv_path.stat().st_size
parquet_kb = parquet_size / 1024
csv_kb = csv_size / 1024

compression_ratio = csv_size / parquet_size

print(f"Parquet size: {parquet_kb:.2f} KB")
print(f"CSV size: {csv_kb:.2f} KB")
print(f"Compression ratio (CSV / Parquet): {compression_ratio:.2f}")

Parquet size: 15975.24 KB
CSV size: 45509.75 KB
Compression ratio (CSV / Parquet): 2.85


A Parquet file is divided into row groups, and within each row group, data is stored by column chunks. At the end of the file, a footer contains metadata about the schema, the location of each column chunk, and statistics (min, max, null count) for each chunk.

Because Parquet stores min/max statistics for each column in each row group, query engines can skip entire chunks of data that do not match a filter condition.
Parquet is columnar. If a query selects only 2 out of 20 columns, only those 2 columns are read from disk.
Parquet stores data in compressed binary format with type information.

TASK2

In [18]:
%time pd.read_parquet("data.parquet")

CPU times: total: 31.2 ms
Wall time: 46.5 ms


,user_id,city,score,active,signup_date,age,sessions,revenue
0,1,New York,43.689490,False,2025-12-11 00:45:44.905779200,42,359,272.618107
1,2,Toronto,97.403462,True,2025-03-03 03:42:39.456900608,28,264,107.594249
2,3,Baku,66.148238,False,2024-10-18 07:37:11.266229760,27,480,372.380026
3,4,Lima,21.993544,False,2025-06-28 14:47:27.135960832,75,330,879.705424
4,5,New York,91.719953,True,2024-01-03 14:53:11.526349568,79,140,730.620245
...,...,...,...,...,...,...,...,...
499995,499996,Toronto,61.531318,False,2025-06-05 20:16:35.967020800,77,5,915.694460
499996,499997,Berlin,74.712924,False,2024-02-26 10:38:03.815138816,38,418,904.013702
499997,499998,Ankara,48.430697,True,2023-11-07 13:36:41.184132352,28,233,423.253865
499998,499999,Baku,89.680757,True,2024-10-29 11:15:45.442146560,23,383,626.845396


In [19]:
%time pd.read_parquet("data.parquet", columns=["score", "age"])

CPU times: total: 15.6 ms
Wall time: 9.46 ms


,score,age
0,43.689490,42
1,97.403462,28
2,66.148238,27
3,21.993544,75
4,91.719953,79
...,...,...
499995,61.531318,77
499996,74.712924,38
499997,48.430697,28
499998,89.680757,23


In [20]:
columns_to_select = ["score", "age"]

In [21]:
%time pd.read_csv("data.csv")[columns_to_select]

CPU times: total: 484 ms
Wall time: 485 ms


,score,age
0,43.689490,42
1,97.403462,28
2,66.148238,27
3,21.993544,75
4,91.719953,79
...,...,...
499995,61.531318,77
499996,74.712924,38
499997,48.430697,28
499998,89.680757,23


Parquet is a column-major,meaning that data is stored column by column rather than row by row. In Task 1, we observed that each row group in Parquet stores column chunks, along with metadata like min/max values, null counts, and compressed size.
This column-chunk layout allows selective reads:

If a query accesses just score and age, Parquet only reads the corresponding column chunks from disk.
CSV, by contrast, is row-major: reading any column requires scanning the entire file.

Column statistics (min/max) stored in the metadata enable the engine to skip row groups that don’t match filter conditions.
This reduces I/O and speeds up queries.

Column chunks compress similar data efficiently. Reading fewer columns means reading less compressed data, further improving speed.

TASK3

In [22]:
import pyarrow as pa
import pyarrow.parquet as pq

%time filtered=pq.read_table("data.parquet",filters=[("age", ">", 50)])

CPU times: total: 31.2 ms
Wall time: 31.2 ms


In [23]:
#step2
arrow_table = pq.read_table("data.parquet")

%time df = pq.read_table("data.parquet").to_pandas()

# Apply the filter in pandas
%time filtered_df = df[df["age"] > 50]

CPU times: total: 156 ms
Wall time: 43.7 ms
CPU times: total: 15.6 ms
Wall time: 11.9 ms


In [24]:
len(filtered_df)

237898

In [25]:
len(filtered)

237898

Number of rows are same for predicate pushdown and post filtering.But in timing predicate pushdown is faster from total timing of to pandas and filtering.

Predicate pushdown means the filter is applied at the file scan level, before data is loaded into memory. In column-major formats like Parquet:
Row groups contain column chunks with min/max statistics.
The query engine checks the statistics against the filter.
Entire row groups that cannot satisfy the filter are skipped.
Only relevant column chunks and rows are read from disk.

Faster because
Less I/O:Only matching data is read from disk.
Less memory usage:Unnecessary rows and columns are never loaded.
Reduced processing:No need to filter millions of irrelevant rows after reading.

In [26]:
#step4
import duckdb
%time result = duckdb.sql("SELECT * FROM 'data.parquet' WHERE age > 50").df()

CPU times: total: 78.1 ms
Wall time: 61.5 ms


This type is more slower than both pyarrow and pandas filtering

TASK4

In [27]:
df2=pd.read_parquet("data.parquet")

import duckdb
result = duckdb.sql("SELECT * FROM 'data.parquet'").df()

In [28]:
#query1
%time city_counts = df2['city'].value_counts()

CPU times: total: 15.6 ms
Wall time: 19.4 ms


In [29]:
%time city_counts2 = duckdb.sql("SELECT city, COUNT(*) AS num_records FROM 'data.parquet' GROUP BY city").df()

CPU times: total: 15.6 ms
Wall time: 18.8 ms


In [30]:
#query2
%time avg_df = (df.groupby("city")["score"].mean().reset_index(name="avg_score").sort_values("avg_score", ascending=False))

CPU times: total: 46.9 ms
Wall time: 28 ms


In [31]:
%time avg_duckdb = duckdb.sql("SELECT city, AVG(score) AS avg_score FROM df GROUP BY city ORDER BY avg_score DESC").df()

CPU times: total: 93.8 ms
Wall time: 17.8 ms


In [32]:
#query3
%time per_df = (((df["active"]) & (df["score"] > 75)).groupby(df["city"]).mean() * 100).reset_index(name="percentage")

CPU times: total: 46.9 ms
Wall time: 26.4 ms


In [33]:
%time per_duckdb = duckdb.sql("SELECT city, 100.0 * SUM(CASE WHEN active AND score > 75 THEN 1 ELSE 0 END) / COUNT(*) AS percentage FROM df GROUP BY city").df()

CPU times: total: 78.1 ms
Wall time: 18.6 ms


In [34]:
#query4
%time top_pandas = df[df.groupby("city")["score"].rank(ascending=False) <= 10].sort_values(["city", "score"], ascending=[True, False])

CPU times: total: 219 ms
Wall time: 216 ms


In [35]:
%time top_duckdb = duckdb.sql("SELECT * FROM ( SELECT *, ROW_NUMBER() OVER (PARTITION BY city ORDER BY score DESC) AS rank FROM df) WHERE rank <= 10").df()

CPU times: total: 422 ms
Wall time: 101 ms


In [36]:
#query5
%time scor_pandas = df.sort_values(["city", "user_id"])

%time scor_pandas["running_total_score"] = scor_pandas.groupby("city")["score"].cumsum()

CPU times: total: 93.8 ms
Wall time: 73.5 ms
CPU times: total: 15.6 ms
Wall time: 20 ms


In [37]:
%time scor_duckdb = duckdb.sql("SELECT *,SUM(score) OVER (PARTITION BY city ORDER BY user_id ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_total_score FROM df").df()

CPU times: total: 359 ms
Wall time: 138 ms


for me pandas was easier to use with functions like groupby,mean.
Duckdb is faster

The difference in performance was most noticeable for the window function queries, particularly the top 10 users per city and the running total of scores per city. These operations involve sorting and partitioning the data, which DuckDB handles very efficiently. For simpler aggregations like counts and averages, the performance difference was relatively small and less significant.

TASK5

In [38]:
#step1
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

df = pd.DataFrame({
    "id": [1, 2, 3, 4],
    "name": ["Alice", "Bob", "Charlie", "Diana"],
    "age": [25, 30, 35, 40],
    "salary": [50000.0, 60000.0, 70000.0, 80000.0],
    "is_manager": [False, True, False, True]
})

arrow_table = pa.Table.from_pandas(df)

print(arrow_table)

pyarrow.Table
id: int64
name: string
age: int64
salary: double
is_manager: bool
----
id: [[1,2,3,4]]
name: [["Alice","Bob","Charlie","Diana"]]
age: [[25,30,35,40]]
salary: [[50000,60000,70000,80000]]
is_manager: [[false,true,false,true]]


In [39]:
#step2
print("Arrow Schema:")
print(arrow_table.schema)

Arrow Schema:
id: int64
name: string
age: int64
salary: double
is_manager: bool
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 817


In [40]:
#step3
pq.write_table(arrow_table, "employees.parquet")

arrow_table_read = pq.read_table("employees.parquet")

print("Read Arrow Table:")
print(arrow_table_read)

Read Arrow Table:
pyarrow.Table
id: int64
name: string
age: int64
salary: double
is_manager: bool
----
id: [[1,2,3,4]]
name: [["Alice","Bob","Charlie","Diana"]]
age: [[25,30,35,40]]
salary: [[50000,60000,70000,80000]]
is_manager: [[false,true,false,true]]


In [41]:
#step4
df_new = arrow_table_read.to_pandas()

print("New DataFrame:")
print(df_new)

print("\nData matches original:")
print(df.equals(df_new))

New DataFrame:
   id     name  age   salary  is_manager
0   1    Alice   25  50000.0       False
1   2      Bob   30  60000.0        True
2   3  Charlie   35  70000.0       False
3   4    Diana   40  80000.0        True

Data matches original:
True


In [42]:
#step5
df_arrow_backend = pd.read_parquet(
    "employees.parquet",
    dtype_backend="pyarrow"
)

print("Arrow-backed pandas dtypes:")
print(df_arrow_backend.dtypes)

print("\nTraditional pandas dtypes:")
print(df.dtypes)

Arrow-backed pandas dtypes:
id             int64[pyarrow]
name          string[pyarrow]
age            int64[pyarrow]
salary        double[pyarrow]
is_manager      bool[pyarrow]
dtype: object

Traditional pandas dtypes:
id              int64
name           object
age             int64
salary        float64
is_manager       bool
dtype: object


Parquet defines how columnar data is stored on disk. Apache Arrow defines how columnar data is represented in memory. Together, they form a powerful pair: Parquet for persistent storage, Arrow for in-memory computation.

Parquet (disk): Stores data efficiently on disk in a columnar layout.
Arrow (memory): Reads Parquet into a fast, columnar, zero-copy in-memory format.
pandas (analysis): Can use Arrow tables directly or via dtype_backend="pyarrow" for efficient in-memory operations.
DuckDB (SQL): Queries Arrow tables or Parquet files directly without copying data.
Flow:
Parquet (disk) → Arrow Table (memory) → pandas (analysis) → DuckDB (SQL queries)